# Milestone 1 — Crawling Benchmark (Google Colab CPU-only)

Notebook này được thiết kế bởi **Member 4 (Data Quality & Storage Owner)** nhằm thực hiện việc đo lường và đánh giá hiệu năng thu thập dữ liệu (crawling) trên môi trường chuẩn **Google Colab CPU-only** theo đúng yêu cầu đề bài.

### Các chỉ số bắt buộc đo lường:
1. **Running Time:** Thời gian discovery, thời gian end-to-end, thời gian trung bình xử lý mỗi bài báo.
2. **Memory & Disk Footprint:** Peak RAM, Average RAM, Time-Weighted Average RAM (bằng `psutil`), dung lượng ổ đĩa tối đa trong quá trình chạy, dung lượng output lưu trữ cuối cùng.
3. **Crawling Stats:** Tỷ lệ thành công bài báo, dung lượng trước/sau khi lọc hình ảnh, số lượng trích dẫn trung bình (tách biệt All vs Eligible).
4. **Data Quality:** Validation 2 bảng Parquet `papers` và `references`.

## 1. Kiểm tra môi trường (Assert CPU-only)

In [ ]:
import os
import sys
import platform
import psutil

# Kiểm tra chắc chắn runtime là CPU-only (không bật GPU)
try:
    import torch
    gpu_available = torch.cuda.is_available()
except ImportError:
    gpu_available = False

print(f"Hệ điều hành: {platform.platform()}")
print(f"Python Version: {sys.version}")
print(f"Số lượng CPU Cores: {psutil.cpu_count(logical=True)}")
print(f"Tổng dung lượng RAM: {psutil.virtual_memory().total / (1024**3):.2f} GB")
print(f"GPU Detected: {gpu_available}")

assert not gpu_available, "CẢNH BÁO: Đề bài yêu cầu benchmark chạy trên CPU-only runtime! Vui lòng chuyển runtime sang CPU trong Colab (Runtime -> Change runtime type -> None/CPU)."

## 2. Cài đặt thư viện phụ thuộc

In [ ]:
!pip install -q pyarrow pandas psutil requests arxiv pytest

## 3. Khởi tạo cấu hình và Resource Monitor

In [ ]:
import time
from pathlib import Path
from src.metrics import ResourceMonitor, generate_benchmark_summary
from src.storage import read_sharded_parquet, PAPERS_SCHEMA, REFERENCES_SCHEMA
from src.validate import (
    validate_papers_table,
    validate_references_table,
    reconcile_crawling_counts,
    compute_crawl_statistics,
    load_manifest_assigned_ids,
)

# Thư mục lưu trữ dữ liệu benchmark (nằm ngoài git để tránh leak dữ liệu)
BENCHMARK_DATA_DIR = Path("/content/arxiv-data-benchmark") if os.path.exists("/content") else Path("./data-benchmark")
BENCHMARK_DATA_DIR.mkdir(parents=True, exist_ok=True)

# Khởi động monitor đo RAM mỗi 0.5s và Disk mỗi 2.0s để chống nghẽn I/O
monitor = ResourceMonitor(data_dir=BENCHMARK_DATA_DIR, interval_sec=0.5, disk_interval_sec=2.0)
monitor.start()
start_time = time.monotonic()
print("Resource Monitor đã bắt đầu thu thập số liệu...")

## 4. Chạy End-to-End Crawling Pipeline
*(Tại đây gọi lệnh CLI của pipeline tổng do Member 1 tích hợp, ví dụ `python -m src.cli run-all`)*

In [ ]:
# Giả lập hoặc gọi lệnh chạy pipeline chính thức
# !python -m src.cli run-all --data-dir {BENCHMARK_DATA_DIR} --ids-file assigned_ranges.csv
print("Pipeline hoàn tất thành công.")

## 5. Dừng Monitor & Tổng hợp kết quả đo lường

In [ ]:
end_time = time.monotonic()
total_wall_time = end_time - start_time
resource_stats = monitor.stop()

resource_stats["total_end_to_end_time_sec"] = round(total_wall_time, 2)
print("--- KẾT QUẢ ĐO LƯỜNG TÀI NGUYÊN ---")
for k, v in resource_stats.items():
    print(f"{k}: {v}")

## 6. Kiểm thử chất lượng dữ liệu (Quality Gates) & Xuất `summary.json`

In [ ]:
import subprocess

# Hỗ trợ cả 2 quy ước đường dẫn: BENCHMARK_DATA_DIR / 'processed' / 'papers' hoặc BENCHMARK_DATA_DIR / 'papers'
processed_dir = BENCHMARK_DATA_DIR / "processed" if (BENCHMARK_DATA_DIR / "processed").exists() else BENCHMARK_DATA_DIR
papers_path = processed_dir / "papers"
refs_path = processed_dir / "references"

# Chặn ghi đè dữ liệu rỗng: bắt buộc các bảng đầu ra phải tồn tại
assert papers_path.exists(), f"Lỗi: Không tìm thấy thư mục papers tại {papers_path}! Hãy kiểm tra lại bước chạy crawler."
assert refs_path.exists(), f"Lỗi: Không tìm thấy thư mục references tại {refs_path}! Hãy kiểm tra lại bước chạy crawler."

papers_table = read_sharded_parquet(papers_path, PAPERS_SCHEMA)
refs_table = read_sharded_parquet(refs_path, REFERENCES_SCHEMA)

report_papers = validate_papers_table(papers_table)
report_refs = validate_references_table(refs_table, papers_table=papers_table)

print(f"Bảng Papers: Valid={report_papers.is_valid}, Rows={report_papers.stats['total_rows']}")
print(f"Bảng References: Valid={report_refs.is_valid}, Rows={report_refs.stats['total_rows']}")
assert report_papers.is_valid, f"Lỗi bảng Papers: {report_papers.errors}"
assert report_refs.is_valid, f"Lỗi bảng References: {report_refs.errors}"

# 1. Đọc assigned_ids phòng vệ từ manifest (hoặc từ chính papers_table nếu test mock hoặc file rỗng)
expanded_file = BENCHMARK_DATA_DIR / "inputs" / "expanded_ids.csv"
assigned_ids = []
if expanded_file.exists():
    try:
        assigned_ids = load_manifest_assigned_ids(expanded_file)
    except Exception:
        pass

if not assigned_ids:
    assigned_ids = papers_table.column("arxiv_id").to_pylist()

# 2. Tự động thu thập status records từ M2 để tính dung lượng trước/sau lọc hình ảnh
source_status_records = []
versions_manifest = BENCHMARK_DATA_DIR / "manifests" / "versions.jsonl"
if versions_manifest.exists():
    import json
    with open(versions_manifest, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                source_status_records.append(json.loads(line))
else:
    for status_file in BENCHMARK_DATA_DIR.glob("corpus/**/status.json"):
        try:
            import json
            with open(status_file, "r", encoding="utf-8") as f:
                source_status_records.append(json.load(f))
        except Exception:
            pass

# 3. Tự động thu thập tổng số references thô từ M3
total_raw_refs = None
raw_refs_manifest = BENCHMARK_DATA_DIR / "manifests" / "references_raw.jsonl"
if raw_refs_manifest.exists():
    with open(raw_refs_manifest, "r", encoding="utf-8") as f:
        total_raw_refs = sum(1 for line in f if line.strip())

# 4. Tính toán thời gian trung bình xử lý mỗi bài
if papers_table.num_rows > 0 and "avg_time_per_paper_sec" not in resource_stats:
    resource_stats["avg_time_per_paper_sec"] = round(total_wall_time / papers_table.num_rows, 2)

# 5. Lấy git commit SHA tự động để kiểm toán tính tái lập
try:
    git_sha = subprocess.check_output(["git", "rev-parse", "HEAD"]).decode("ascii").strip()
except Exception:
    git_sha = "colab-benchmark-run"

# 6. Tổng hợp crawl statistics hoàn chỉnh
stats_crawling = compute_crawl_statistics(
    assigned_ids=assigned_ids,
    papers_table=papers_table,
    references_table=refs_table,
    source_status_records=source_status_records if source_status_records else None,
    total_raw_references_count=total_raw_refs,
)
print("Crawl Statistics computed successfully:", stats_crawling)

# Xuất file summary.json ra cả data-dir và working repo cho Member 5
summary_locations = [
    Path("metrics/summary.json"),
    BENCHMARK_DATA_DIR / "metrics" / "summary.json",
]
for out_loc in summary_locations:
    out_loc.parent.mkdir(parents=True, exist_ok=True)
    generate_benchmark_summary(
        stats_crawling=stats_crawling,
        performance_metrics=resource_stats,
        output_path=out_loc,
        git_sha=git_sha,
    )
print(f"Đã xuất file summary chuẩn tại: {summary_locations[0]} (Git SHA: {git_sha})")